# PDF Retrival

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data.pdf")
pages = loader.load_and_split()

/tmp/ipykernel_117020/1905935572.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
len(pages)

15

In [4]:
for i, p in enumerate(pages, start=1):
    print(f"Page {i} metadata: {p.metadata}")
    print(p.page_content[:200])  # print first 200 chars for a quick preview
    print("---")

Page 1 metadata: {'producer': 'Skia/PDF m149', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36', 'creationdate': '2026-06-08T14:48:17+00:00', 'title': 'How LLMs Work: Tokens, Embeddings, and Transformers | by Abdul Moiz | Apr, 2026 | Medium', 'moddate': '2026-06-08T14:48:17+00:00', 'source': 'data.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}
How LLMs Work: Tokens, Embeddings, and
Transformers
11 min read · Apr 23, 2026
Abdul Moiz Follow
Listen Share
Series: LLM Engineering from the Ground Up | Article 1 of 10
Hey everyone !! back again wi
---
Page 2 metadata: {'producer': 'Skia/PDF m149', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36', 'creationdate': '2026-06-08T14:48:17+00:00', 'title': 'How LLMs Work: Tokens, Embeddings, and Transformers | by Abdul Moiz | Apr, 2026 | Medium', 'moddate': '2026-06-08T14:48:17+00:00', 'sourc

# WebPage Retrival

In [1]:
from langchain_community.document_loaders import WebBaseLoader

URL = "https://www.theverge.com/2024/4/18/24133808/meta-ai-assistant-llama-3-chatgpt-openai-rival"
loader = WebBaseLoader(URL)
data = loader.load()

/tmp/ipykernel_279725/3810729084.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
/home/ajayp/anaconda3/envs/langchain/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
len(data)
# typically 1 for standard article pages

doc = data[0]
print(type(doc))                # e.g., <class 'langchain.schema.Document'>
print(doc.metadata)             # metadata dictionary (may include title, source url)
print(doc.page_content[:300])   # print the first 300 characters of the page

<class 'langchain_core.documents.base.Document'>
{'source': 'https://www.theverge.com/2024/4/18/24133808/meta-ai-assistant-llama-3-chatgpt-openai-rival', 'title': 'Meta releases new AI assistant powered by Llama 3 model | The Verge', 'description': 'Meta released Llama 3 and is expanding access to the Meta AI bot. Meta CEO Mark Zuckerberg says the company has built “the most intelligent AI assistant” available for free.', 'language': 'en-US'}
Meta releases new AI assistant powered by Llama 3 model | The VergeSkip to main contentThe homepageThe VergeThe Verge logo.The VergeThe Verge logo.TechReviewsScienceEntertainmentAIPolicyNotificationsNotificationsHamburger Navigation ButtonThe homepageThe VergeThe Verge logo.NotificationsNotification


# Pass Document as Context

In [4]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        ("human", "Answer this {question}, here is some extra {context}"),
    ]
)

messages = chat_template.format_messages(
    name="Website",
    question="Tell me summery",
    context=data
)

llm = ChatOpenAI(
    api_key=os.getenv("KODEKEY_API_KEY"),
    base_url=os.getenv("KODEKEY_API_URL"),
    model="deepseek/deepseek-v4-flash"
)

response = llm.invoke(messages)
print(response.content)

**Summary of the article:**  
Meta released **Llama 3** and is expanding its **Meta AI assistant**, positioning it as a free competitor to ChatGPT. The assistant is being integrated across **Instagram, Facebook, WhatsApp, Messenger**, Facebook’s main feed, search bars, and a new standalone site, **Meta.ai**. It is rolling out in English beyond the US to countries including Canada, Australia, Nigeria, Pakistan, Singapore, and others.

**Key points:**
- Meta released two open-source Llama 3 models: **8B and 70B parameters**, available to developers and inside Meta AI. A much larger, multimodal version with **over 400B parameters** is coming later.
- Meta claims Llama 3 beats competing models of its class on benchmarks and is better at coding. It trained on over **15 trillion tokens**, compared with Llama 2’s 2 trillion, using public internet data and synthetic AI data — but no Meta user data.
- Meta AI now integrates **real-time search results from both Bing and Google**, upgraded image 

# Chunking Document

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
chunks = text_splitter.split_documents(pages)

In [13]:
len(chunks)
# -> 40
chunks[0]
# -> Document(page_content='LakeSide Bicycles Employee Handbook Welcome to the team! LakeSide Bicycles is a company that values quality, innovation, and customer satisfaction. We are passionate about creating and selling', metadata={'source': 'data/handbook.pdf', 'page': 0})

Document(metadata={'producer': 'Skia/PDF m149', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36', 'creationdate': '2026-06-08T14:48:17+00:00', 'title': 'How LLMs Work: Tokens, Embeddings, and Transformers | by Abdul Moiz | Apr, 2026 | Medium', 'moddate': '2026-06-08T14:48:17+00:00', 'source': 'data.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='How LLMs Work: Tokens, Embeddings, and\nTransformers\n11 min read · Apr 23, 2026\nAbdul Moiz Follow\nListen Share\nSeries: LLM Engineering from the Ground Up | Article 1 of 10')

# Embedding

In [19]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
docs = [
    "Thrilling Finale Awaits: The Countdown to the Cricket World Cup Championship",
    "Global Giants Clash: Football World Cup Semi-Finals Set the Stage for Epic Showdowns",
    "Record Crowds and Unforgettable Moments: Highlights from the Cricket World Cup",
    "From Underdogs to Contenders: Football World Cup Surprises and Breakout Stars"
]
embed_docs = embeddings.embed_documents(docs)
len(embed_docs)

/home/ajayp/anaconda3/envs/langchain/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1751.58it/s]


4

# Performing Semantic Search

In [32]:
from langchain_chroma import Chroma
vectorstore = Chroma.from_texts(texts=docs, embedding=embeddings)

In [33]:
# Example: retrieve top 2 documents most related to "Rohit Sharma"
results_rohit = vectorstore.similarity_search("Rohit Sharma", k=2)
print(results_rohit)
# Example output (list of Document objects):
# [Document(page_content='Record Crowds and Unforgettable Moments: Highlights from the Cricket World Cup'),
# Example: retrieve top 2 documents most related to "Lionel Messi"
results_messi = vectorstore.similarity_search("Lionel Messi", k=2)
print(results_messi)
# Example output:
# [Document(page_content='From Underdogs to Contenders: Football World Cup Surprises and Breakout Stars'),
#  Document(page_content='Global Giants Clash: Football World Cup Semi-Finals Set the Stage for Epic Showdowns')]

[Document(id='5d82eec6-3307-4d53-b2c2-e15c860b4af6', metadata={}, page_content='Thrilling Finale Awaits: The Countdown to the Cricket World Cup Championship'), Document(id='8e7f8c65-82e7-4f38-b419-5695fe924061', metadata={}, page_content='Record Crowds and Unforgettable Moments: Highlights from the Cricket World Cup')]
[Document(id='06c2726f-dabc-44a6-8f2a-3f8504224a2d', metadata={}, page_content='From Underdogs to Contenders: Football World Cup Surprises and Breakout Stars'), Document(id='8e7f8c65-82e7-4f38-b419-5695fe924061', metadata={}, page_content='Record Crowds and Unforgettable Moments: Highlights from the Cricket World Cup')]
